# 🔧 Polars: Memory-Efficient Parquet Merge
**88M+ Rows with 12GB RAM - Zero OOM Errors**

Bu script:
- `scan_parquet()` ile lazy metadata okuma
- `sink_parquet()` ile streaming disk yazımı
- RAM spike'sız out-of-core işleme

In [ ]:
!pip install polars -q

import polars as pl
import os
from pathlib import Path

print(f'Polars version: {pl.__version__}')

# RAM kontrolü
import psutil
print(f'Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB')

In [ ]:
# Google Drive bağlantısı
from google.colab import drive
drive.mount('/content/drive')

# Paths - KENDİ YOLLARINI GÜNCELLE
INPUT_PATTERN = '/content/drive/MyDrive/DiabetesKDS/data/*.parquet'  # Glob pattern
OUTPUT_FILE = '/content/drive/MyDrive/DiabetesKDS/data/merged_dataset.parquet'

## 1. Lazy Scan (RAM = 0)

In [ ]:
# Lazy scan - RAM'e HİÇBİR ŞEY yüklenmez
# Sadece dosya metadata'sı okunur

lf = pl.scan_parquet(INPUT_PATTERN)

# Schema kontrolü
print('Schema (RAM kullanmadan):')
for col, dtype in lf.collect_schema().items():
    print(f'  {col}: {dtype}')

print(f'\n✅ LazyFrame oluşturuldu - RAM hala boş!')
print(f'   Mevcut RAM: {psutil.virtual_memory().available / 1e9:.1f} GB')

## 2. Opsiyonel: Type Optimization

Float64 → Float32 ve Int64 → Int32 dönüşümü RAM'i %50 azaltır.

In [ ]:
# Type downcasting (opsiyonel ama önerilen)
def optimize_types(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Cast to smaller types for memory efficiency."""
    schema = lf.collect_schema()
    
    casts = []
    for col, dtype in schema.items():
        if dtype == pl.Float64:
            casts.append(pl.col(col).cast(pl.Float32))
        elif dtype == pl.Int64:
            casts.append(pl.col(col).cast(pl.Int32))
        # Diğer türler olduğu gibi kalır
    
    if casts:
        return lf.with_columns(casts)
    return lf

# Uygula
lf = optimize_types(lf)
print('✅ Type optimization uygulandı (lazy - hala RAM = 0)')

## 3. Streaming Sink to Parquet

**KRİTİK:** `.collect()` KULLANMA! Bu tüm veriyi RAM'e yükler.

`.sink_parquet()` chunk chunk işler ve diske yazar.

In [ ]:
%%time

print('='*60)
print('STREAMING MERGE BAŞLIYOR')
print('='*60)
print(f'Output: {OUTPUT_FILE}')
print('\n⚠️ Bu işlem veri boyutuna göre 10-60 dakika sürebilir...')
print('   RAM kullanımı düşük kalacak (~2-4 GB)\n')

# STREAMING SINK - RAM spike'sız
lf.sink_parquet(
    OUTPUT_FILE,
    compression='zstd',      # Yüksek sıkıştırma
    row_group_size=100_000,  # Chunk boyutu
)

print('\n✅ MERGE TAMAMLANDI!')

# Dosya boyutunu kontrol et
size_gb = os.path.getsize(OUTPUT_FILE) / (1024**3)
print(f'   Çıktı boyutu: {size_gb:.2f} GB')

## 4. Verification (RAM-Safe)

In [ ]:
# Tüm dosyayı yüklemeden ilk 5 satırı al
print('İlk 5 satır (RAM-safe fetch):')

sample = pl.scan_parquet(OUTPUT_FILE).fetch(5)
print(sample)

# Toplam satır sayısı (lazy count)
total_rows = pl.scan_parquet(OUTPUT_FILE).select(pl.len()).collect().item()
print(f'\n📊 Toplam satır: {total_rows:,}')

In [ ]:
# RAM durumu
print(f'\n💾 Final RAM durumu: {psutil.virtual_memory().available / 1e9:.1f} GB boş')
print('\n🎉 88M+ satır başarıyla merge edildi - OOM yok!')